# SSD Basic 配對交易滾動回測系統：完整交易邏輯分析
### （Gatev et al. 2006 距離法實作）

---

## 目錄

1. [系統架構總覽](#1-系統架構總覽)
2. [Phase 1：資料前處理（DataProcessor）](#2-phase-1資料前處理dataprocessor)
3. [Phase 2：形成期（Formation）](#3-phase-2形成期formation)
4. [Phase 3：交易期（Trading）](#4-phase-3交易期trading)
5. [Phase 4：滾動回測引擎（RollingBacktester）](#5-phase-4滾動回測引擎rollingbacktester)
6. [Phase 5：資金管理與結果匯出](#6-phase-5資金管理與結果匯出)
7. [完整公式彙整表](#7-完整公式彙整表)
8. [設計決策說明](#8-設計決策說明)

---

## 1. 系統架構總覽

本系統實作 Gatev, Do, Gatev & McLean (2006)「距離法（Distance Approach）」配對交易策略。與回歸法（Cointegration/OLS）的核心差異在於：**直接比較股票的累積總回報指數走勢，避險比率恆定為 1:1 等額美元中性（Dollar-Neutral），不估計任何 OLS Beta**。

```
┌─────────────────────────────────────────────────────┐
│                   RollingBacktester                  │
│   (滾動視窗排程 + 網格搜尋 Grid Search)               │
│                                                     │
│  ┌──────────────┐      ┌──────────────────────────┐ │
│  │ DataProcessor│ ───> │  Formation（形成期模組）  │ │
│  │ 資料前處理   │      │  - 首日歸一化（≠ Z-Score） │ │
│  └──────────────┘      │  - SSD 最小配對篩選        │ │
│                        │  - ADF 共整合檢驗           │ │
│                        │  - OU 半衰期過濾             │ │
│                        │  - Hurst 指數篩選            │ │
│                        └────────────┬─────────────┘ │
│                                     │ selected_pairs │
│                        ┌────────────▼─────────────┐ │
│                        │  Trading（交易期模組）    │ │
│                        │  - 首日價格延伸正規化      │ │
│                        │  - Z-Score 計算（β≡1.0）  │ │
│                        │  - 等額資金分配（50/50）   │ │
│                        │  - 進出場邏輯 / 停損機制   │ │
│                        └──────────────────────────┘ │
└─────────────────────────────────────────────────────┘
```

**滾動視窗時間軸示意（formation=252, trading=126, step=21）：**

```
時間軸 ──────────────────────────────────────────────────────────────────►

期1: [════════════ 形成期 252天 ════════════│════════ 交易期 126天 ════════]
期2:       [════════════ 形成期 252天 ════════════│════════ 交易期 126天 ════════]
期3:             [════════════ 形成期 252天 ════════════│════════ 交易期 126天 ════════]
...
期6:                               [════════════ 形成期 252天 ════════════│════════ 交易期 126天 ════════]
                 ◄─ step=21 ─►

※ 每期僅錯開 rolling_step=21 天，交易期大量重疊並行（最多同時 6 期）
※ 並行期數上限：max_concurrent = trading_window // rolling_step = 126 // 21 = 6
※ 資金切為 6 槽輪流承接，每槽在前一期交易期結束後才釋放供下一期使用
```

---

## 2. Phase 1：資料前處理（DataProcessor）

與前版本完全相同，執行四道清洗：

**Step A：移除高缺失值標的**

$$\text{保留條件：} \frac{\text{NaN 筆數}}{\text{總交易日數}} < 0.20$$

**Step B：向前填補（Forward Fill）**

$$P_t = P_{t-k}, \quad k = \min(\text{連續缺失天數},\ 5)$$

**Step C：移除高缺失值日期**

$$\text{保留條件：} \frac{\text{該日缺失標的數}}{\text{總標的數}} \leq 0.10$$

**Step D：移除覆蓋率不足標的**

$$\text{保留條件：非 NaN 筆數} \geq \lfloor N_{total} \times 0.90 \rfloor$$

滾動視窗起始延伸公式：

$$\text{data\_slice\_start} = \text{all\_dates}[\max(0,\ \text{first\_trade\_idx} - W_F)]$$

---

## 3. Phase 2：形成期（Formation）

### 3.1 正規化方式：首日歸一（Cumulative Total Returns Index）

**本版本與前版本（Z-Score 正規化）的核心差異在此。**

直接以各股票在形成期第一個交易日的收盤價作為分母，將價格序列轉換為「累積總回報指數」，令第一天索引值恆等於 1.0：

$$I_{i,t} = \frac{P_{i,t}}{P_{i,1}}, \quad I_{i,1} = 1.0$$

其中 $P_{i,1}$ 為股票 $i$ 在形成期第一日的收盤價（若 $P_{i,1} \leq 10^{-8}$ 則設為 1.0 防範除以零）。

**重要特性：**

- 所有股票在形成期第一天索引值強制對齊為 1.0
- 保留絕對量級資訊（不像 Z-Score 消除均值與標準差）
- SSD 衡量的是「兩條累積回報曲線的實際面積差異」，對「回報幅度不同但走勢相似」的股票仍會給出較大的 SSD

### 3.2 SSD 計算

$$\text{SSD}(A, B) = \sum_{t=1}^{T}\left(I_{A,t} - I_{B,t}\right)^2$$

等價於：

$$\text{SSD}(A, B) = \|\mathbf{I}_A - \mathbf{I}_B\|^2$$

**實作：** 使用 `scipy.spatial.distance.pdist(norm_vals, metric='sqeuclidean')` 向量化計算。

### 3.3 避險比率（Hedge Ratio）

**本版本不估計 OLS Beta，固定為：**

$$\hat{\beta} \equiv 1.0$$

這是距離法的核心假設：以等美元名義金額（Equal Dollar Weighting）持有兩腳部位。在 `compute_ssd()` 中以 `beta = 1.0` 直接賦值，不做任何協方差/方差的估計。

### 3.4 價差（Spread）計算

$$S_t = I_{A,t} - 1.0 \times I_{B,t} = I_{A,t} - I_{B,t}$$

### 3.5 統計篩選三關卡

SSD 最小的前 $\max(200,\ \text{top\_n} \times 15)$ 組候選配對，依序通過以下三個檢驗（邏輯與前版本相同）：

---

#### 關卡 A：ADF 共整合檢驗（Augmented Dickey-Fuller Test）

對價差序列 $S_t = I_{A,t} - I_{B,t}$ 進行 ADF 檢定（`regression="n"`，無截距）：

$$\Delta S_t = \rho S_{t-1} + \sum_{k=1}^{p} \gamma_k \Delta S_{t-k} + \varepsilon_t$$

**篩選條件：** $p\text{-value} < 0.05$

---

#### 關卡 B：Ornstein-Uhlenbeck 半衰期

對 $S_t$ 做含截距的 OLS 迴歸：

$$\Delta S_t = \alpha + \lambda S_{t-1} + \varepsilon_t$$

**半衰期：**

$$\tau_{1/2} = -\frac{\ln 2}{\lambda}$$

**篩選條件：**

$$\lambda < 0 \quad \text{且} \quad 2.0 \leq \tau_{1/2} \leq 40.0 \text{ 天}$$

---

#### 關卡 C：Hurst 指數篩選

R/S 分析，對三個分段長度 $n \in \{T/4,\ T/2,\ T\}$ 計算：

$$R/S(n) = \frac{\max_{1 \leq k \leq n}\sum_{i=1}^{k}(\Delta S_i - \overline{\Delta S}) - \min_{1 \leq k \leq n}\sum_{i=1}^{k}(\Delta S_i - \overline{\Delta S})}{\sigma_{\Delta S,n}}$$

以 OLS 擬合斜率得 Hurst 指數：

$$\ln(R/S) = H \cdot \ln(n) + C \implies H = \frac{d\ln(R/S)}{d\ln(n)}$$

**篩選條件：** $H < 0.40$

---

### 3.6 形成期輸出

通過三關卡後，記錄以下統計量：

$$\mu_S = \frac{1}{T}\sum_{t=1}^{T}S_t, \quad \sigma_S = \sqrt{\frac{1}{T-1}\sum_{t=1}^{T}(S_t - \mu_S)^2}$$

以及形成期**第一日原始股價**（非正規化值），供交易期使用：

$$P_{A,1}^{form},\quad P_{B,1}^{form}$$

輸出欄位：`Form_Start`, `Form_End`, `Sector`, `Ticker_A`, `Ticker_B`, `SSD`, `Hedge_Ratio`（固定 1.0）, `Spread_Mean` ($\mu_S$), `Spread_Std` ($\sigma_S$), `First_Price_A`, `First_Price_B`

---

## 4. Phase 3：交易期（Trading）

### 4.1 交易期正規化：延用形成期首日基準價

**這是本版本最關鍵的設計：交易期的正規化基準沿用形成期第一天的收盤價，而非交易期自身的起始價。**

$$I_{A,t}^{trade} = \frac{P_{A,t}}{P_{A,1}^{form}}, \quad I_{B,t}^{trade} = \frac{P_{B,t}}{P_{B,1}^{form}}$$

此設計確保形成期與交易期的正規化基準完全一致，Z-Score 計算不會因換期而產生跳躍。

### 4.2 Z-Score 計算模式

系統支援兩種模式，但兩者的 $\beta$ 均固定為 1.0：

---

#### 模式一：固定參數（`zscore_window = 0`）

**價差：**

$$S_t = I_{A,t}^{trade} - I_{B,t}^{trade}$$

**Z-Score（使用形成期的 $\mu_S$、$\sigma_S$）：**

$$Z_t = \text{clip}\!\left(\frac{S_t - \mu_S}{\max(\sigma_S,\ \sigma_{min})},\ -Z_{clip},\ +Z_{clip}\right)$$

---

#### 模式二：滾動視窗（`zscore_window = W > 0`）

**注意：本版本滾動模式的 Beta 固定為 1.0，不做滾動 Beta 估計。**

滾動均值差（rolling alpha，即滾動截距）：

$$\hat{\alpha}_t^{roll} = \bar{I}_{A,W} - \bar{I}_{B,W}$$

其中 $\bar{I}_{A,W}$、$\bar{I}_{B,W}$ 分別為前 $W$ 日的滾動均值。

**去截距後的滾動價差：**

$$S_t^{roll} = I_{A,t}^{trade} - \hat{\alpha}_t^{roll} - I_{B,t}^{trade}$$

展開後等價於：

$$S_t^{roll} = (I_{A,t}^{trade} - I_{B,t}^{trade}) - (\bar{I}_{A,W} - \bar{I}_{B,W})$$

即「當日的累積回報差」減去「近 $W$ 日的平均累積回報差」，意義為「相對近期均值的偏離程度」。

**滾動標準差（對原始差序列，而非殘差）：**

$$\sigma_{S,t}^{roll} = \text{std}_W(I_{A,t}^{trade} - I_{B,t}^{trade})$$

**Z-Score（滾動模式）：**

$$Z_t = \text{clip}\!\left(\frac{S_t^{roll}}{\max(\sigma_{S,t}^{roll},\ \sigma_{min})},\ -Z_{clip},\ +Z_{clip}\right)$$

---

#### 選配：波動度調整（`use_vol_adjust = True`）

兩種模式均支援此選項，在基礎標準差上乘以波動度膨脹因子：

$$\text{vol\_factor}_t = \max\!\left(1.0,\ \frac{\sigma_{S,20d}}{\sigma_S^{form}}\right)$$

$$\sigma_{adj,t} = \max(\sigma_t \cdot \text{vol\_factor}_t,\ \sigma_{min})$$

### 4.3 資金分配：等額美元中性（1:1 Dollar-Neutral）

**本版本固定 50/50 等額分配，不依 Beta 加權：**

$$V_A = V_B = \frac{C}{2}$$

其中 $C$ 為 `capital_per_pair`。

**股數計算：**

$$N_A = \frac{V_A}{P_A} = \frac{C/2}{P_A}, \quad N_B = \frac{V_B}{P_B} = \frac{C/2}{P_B}$$

### 4.4 進場邏輯（Entry）

**Short Spread（做空價差，$Z_t > Z_{entry}$）：**

意義：$A$ 累積回報相對 $B$ 過高，預期兩者回報差縮小。

$$\text{Position} = -1:\quad N_A^{pos} = -\frac{C/2}{P_A}\ (\text{放空 A}),\quad N_B^{pos} = +\frac{C/2}{P_B}\ (\text{做多 B})$$

**Long Spread（做多價差，$Z_t < -Z_{entry}$）：**

意義：$A$ 累積回報相對 $B$ 過低，預期差值回歸。

$$\text{Position} = +1:\quad N_A^{pos} = +\frac{C/2}{P_A}\ (\text{做多 A}),\quad N_B^{pos} = -\frac{C/2}{P_B}\ (\text{放空 B})$$

**冷卻方向機制（Cooldown Direction）：**

平倉後記錄方向 `cooldown_dir`，防止在訊號未完全回歸前立即反向再進場：

- `cooldown_dir = -1`：等 $Z_t \leq Z_{exit}$ 後解除
- `cooldown_dir = +1`：等 $Z_t \geq -Z_{exit}$ 後解除

### 4.5 未實現損益（Unrealized PnL）

$$\text{UnrPnL}_t = N_A^{pos}(P_{A,t} - P_A^{entry}) + N_B^{pos}(P_{B,t} - P_B^{entry}) - F_{entry} - F_{exit}^{est}$$

**交易成本（手續費 + 滑價）：**

$$F_{entry} = \left(|N_A^{pos}| \cdot P_A^{entry} + |N_B^{pos}| \cdot P_B^{entry}\right) \cdot r_{friction}$$

$$F_{exit}^{est} = \left(|N_A^{pos}| \cdot P_{A,t} + |N_B^{pos}| \cdot P_{B,t}\right) \cdot r_{friction}$$

$$r_{friction} = r_{fee} + r_{slippage}$$

### 4.6 平倉邏輯（Exit）

**正常平倉條件：**

- Short Spread 平倉：$Z_t \leq Z_{exit}$（價差縮回正常範圍）
- Long Spread 平倉：$Z_t \geq -Z_{exit}$（價差縮回正常範圍）

**已實現損益：**

$$\text{ClosedPnL} = N_A^{pos}(P_{A,t}^{exit} - P_A^{entry}) + N_B^{pos}(P_{B,t}^{exit} - P_B^{entry}) - F_{entry} - F_{exit}^{actual}$$

$$\text{RealizedPnL} \mathrel{+}= \text{ClosedPnL}$$

### 4.7 停損機制（三層）

#### 層級 1：個別配對資金停損

$$\text{觸發條件：} \frac{-\text{UnrPnL}_t}{C} \geq \text{stop\_loss\_pct}$$

#### 層級 2：動態 Z-Score 停損

$$\text{觸發條件：} |Z_t| > Z_{dyn\_stop}$$

#### 層級 3：投資組合總體停損（後置斷路器）

每日掃描所有配對的累積損益總和：

$$\text{Portfolio\_PnL}_t = \sum_{i=1}^{N} \text{CumPnL}_{i,t}, \quad C_{total} = C \times N$$

$$\text{觸發條件：} \frac{\text{Portfolio\_PnL}_t}{C_{total}} \leq -\text{portfolio\_stop\_loss\_pct}$$

觸發後：$\forall t > t_{cutoff}$，所有配對強制平倉並凍結損益。

### 4.8 期末強制平倉

交易期最後一日若仍有未平倉部位，以當日收盤價強制結算：

$$\text{FinalClosedPnL} = N_A^{pos}(P_{A,T} - P_A^{entry}) + N_B^{pos}(P_{B,T} - P_B^{entry}) - F_{entry} - F_{exit}$$

### 4.9 每日損益變化

$$\Delta_t = \text{CumPnL}_t - \text{CumPnL}_{t-1}$$

---

## 5. Phase 4：滾動回測引擎（RollingBacktester）

與前版本結構完全相同，差異僅在傳入 Trading 的參數內容。

### 5.1 滾動視窗排程

第 $k$ 期：

$$\text{形成期：} [t_k - W_F,\ t_k), \quad \text{交易期：} [t_k,\ t_k + W_T)$$

$$t_{k+1} = t_k + \Delta, \quad N_{slots} = \left\lfloor \frac{W_T}{\Delta} \right\rfloor$$

### 5.2 延伸交易資料（Extended Trade Data）

當 `zscore_window = W > 0` 時，往前延伸以確保滾動計算有足夠的暖機資料：

$$\text{extended\_start\_idx} = \max(0,\ t_k - \max(W_{list}))$$

延伸資料僅用於 Z-Score 計算，不產生交易紀錄。

### 5.3 網格搜尋（Grid Search）

笛卡兒積遍歷以下參數：

| 參數 | 說明 |
|------|------|
| `top_n` | 選取最佳配對數 |
| `stop_loss_pct` | 個別停損比率 |
| `zscore_window` | Z-Score 滾動視窗（0 = 固定） |
| `portfolio_stop_loss_pct` | 組合總體停損 |
| `max_sector_ratio` | 單一產業配對上限比率 |
| `dynamic_stop_z` | 動態 Z-Score 停損閾值 |
| `use_vol_adjust` | 波動度調整開關 |

$$N_{combos} = \prod_{p \in \text{params}} |\text{p\_list}|$$

### 5.4 產業分散化過濾

$$\text{max\_per\_sector} = \max\!\left(1,\ \lfloor \text{top\_n} \times \text{sec\_ratio} \rfloor\right)$$

---

## 6. Phase 5：資金管理與結果匯出

### 6.1 每配對資金分配

$$C_{pair} = \frac{C_{slot}}{N_{pairs}}$$

每槽資金在期末更新：

$$C_{slot}^{next} = \max(0,\ C_{slot} + \text{PeriodPnL})$$

### 6.2 期間損益彙總

$$\text{PeriodPnL} = \sum_{i=1}^{N_{pairs}} \sum_{t \in \text{交易期}} \Delta_{i,t}$$

### 6.3 交易紀錄欄位

| 欄位 | 說明 |
|------|------|
| `Date` | 交易日期 |
| `Price_A / Price_B` | 當日原始收盤價 |
| `Hedge_Ratio` | 固定為 1.0（等額美元中性） |
| `ZScore` | 當日 Z-Score $Z_t$ |
| `Position` | 持倉方向（+1 / -1 / 0） |
| `Unrealized_PnL` | 未實現損益（以原始股價計算） |
| `Realized_PnL` | 累計已實現損益 |
| `Cumulative_PnL` | 總損益（含未實現） |
| `Status` | 當日狀態碼 |
| `Trade_PnL` | 本次交易損益（平倉時填入） |
| `Days_Held` | 持倉天數 |
| `Daily_Delta` | 每日損益變化 $\Delta_t$ |
| `First_Price_A/B` | 形成期首日基準價（用於追溯正規化基準） |

**Status 狀態碼：**

| Status | 觸發條件 |
|--------|---------|
| `HOLD_CASH` | 無持倉，$|Z_t| \leq Z_{entry}$ |
| `HOLD_CASH (COOLDOWN)` | 冷卻方向限制中 |
| `ENTER_LONG_A` | $Z_t < -Z_{entry}$，做多 A 放空 B |
| `ENTER_SHORT_A` | $Z_t > Z_{entry}$，放空 A 做多 B |
| `HOLDING` | 持倉中，未達平倉或停損條件 |
| `EXIT` | $Z_t$ 回歸至 $\pm Z_{exit}$ 以內，正常平倉 |
| `STOP_LOSS_TRIGGERED` | 個別配對停損或動態 Z 停損 |
| `PERIOD_END_EXIT` | 交易期末強制平倉 |
| `PORTFOLIO_STOP_TRIGGERED` | 組合總體停損觸發日（持倉中強制清倉） |
| `PORTFOLIO_STOPPED` | 組合停損觸發日（無持倉，狀態標記） |
| `STOPPED` | 停損後永久停止（或組合停損後續日） |

---

## 7. 完整公式彙整表

### 形成期

| 名稱 | 公式 |
|------|------|
| 累積回報指數 | $I_{i,t} = P_{i,t} / P_{i,1}$ |
| 價差 | $S_t = I_{A,t} - I_{B,t}$ |
| SSD | $\text{SSD}(A,B) = \sum_t (I_{A,t} - I_{B,t})^2$ |
| 避險比率 | $\hat{\beta} \equiv 1.0$（不估計） |
| ADF 迴歸 | $\Delta S_t = \rho S_{t-1} + \sum_k \gamma_k \Delta S_{t-k} + \varepsilon_t$ |
| OU 迴歸 | $\Delta S_t = \alpha + \lambda S_{t-1} + \varepsilon_t$ |
| 半衰期 | $\tau_{1/2} = -\ln 2 / \lambda$ |
| Hurst 指數 | $H = d\ln(R/S) / d\ln(n)$ |
| 形成期均值/標準差 | $\mu_S,\ \sigma_S$ |

### 交易期

| 名稱 | 公式 |
|------|------|
| 交易期正規化 | $I_{i,t}^{trade} = P_{i,t} / P_{i,1}^{form}$ |
| 固定 Z-Score | $Z_t = \text{clip}\left((S_t - \mu_S)/\max(\sigma_S, \sigma_{min}),\ \pm Z_{clip}\right)$ |
| 滾動截距 | $\hat{\alpha}_t^{roll} = \bar{I}_{A,W} - \bar{I}_{B,W}$ |
| 滾動價差 | $S_t^{roll} = (I_{A,t} - I_{B,t}) - (\bar{I}_{A,W} - \bar{I}_{B,W})$ |
| 滾動 Z-Score | $Z_t = \text{clip}\left(S_t^{roll} / \max(\sigma_{S,t}^{roll}, \sigma_{min}),\ \pm Z_{clip}\right)$ |
| 資金分配（A） | $V_A = C / 2$ |
| 資金分配（B） | $V_B = C / 2$ |
| 股數 | $N_i = V_i / P_i$ |
| 未實現損益 | $\text{UnrPnL}_t = \sum_i N_i^{pos}(P_{i,t} - P_i^{entry}) - F_{entry} - F_{exit}^{est}$ |
| 交易成本 | $F = (|N_A|P_A + |N_B|P_B) \cdot (r_{fee} + r_{slippage})$ |
| 個別停損 | $-\text{UnrPnL}_t / C \geq \text{SL\_pct}$ |
| 動態 Z 停損 | $|Z_t| > Z_{dyn}$ |
| 組合停損 | $\sum_i \text{CumPnL}_{i,t} / C_{total} \leq -\text{PSL\_pct}$ |
| 每日 Delta | $\Delta_t = \text{CumPnL}_t - \text{CumPnL}_{t-1}$ |
| 期間損益 | $\text{PeriodPnL} = \sum_{i,t} \Delta_{i,t}$ |
| 資金複利 | $C_{next} = \max(0, C_{curr} + \text{PeriodPnL})$ |

---

## 8. 設計決策說明

### 8.1 距離法 vs. 協整法的本質差異

| 面向 | 本版本（距離法） | 前版本（協整法） |
|------|----------------|----------------|
| 正規化方式 | 首日歸一（累積回報指數） | Z-Score（去均值去標準差） |
| 避險比率 | 固定 $\hat{\beta} = 1.0$ | OLS 估計 $\hat{\beta} = \text{Cov}/\text{Var}$ |
| 資金分配 | 50 / 50 等額 | 依 $\hat{\beta}$ 比例加權 |
| SSD 的意義 | 兩條回報曲線的面積差異 | 兩條正規化曲線的幾何距離 |
| 交易期延伸正規化 | 沿用形成期首日基準 $P_{i,1}^{form}$ | 沿用形成期 $\bar{\ell}_i$, $\sigma_{\ell_i}$ |

### 8.2 滾動 Z-Score 模式（本版本）與前版本的差異

本版本（距離法）滾動模式：$\hat{\beta} \equiv 1.0$，標準差直接對差序列計算：

$$\sigma_{S,t}^{roll,\text{distance}} = \text{std}_W(I_{A,t}^{trade} - I_{B,t}^{trade})$$

本版本的滾動標準差計算更為簡單直接，不涉及殘差推導。

### 8.3 「先初篩再過濾」效能優化

SSD 排序後僅對前 $\max(200,\ \text{top\_n} \times 15)$ 組候選進行慢速統計檢驗（ADF、OU、Hurst），使統計迴歸次數從 $O(N^2)$ 降至常數級，速度提升 30~50 倍。

### 8.4 首日基準價的傳遞機制

形成期輸出 `First_Price_A` / `First_Price_B`（原始絕對價格），交易期接收後用於對交易期任意日期的收盤價做正規化。此機制確保：

1. 形成期與交易期使用同一個正規化基準點
2. 即使換期，交易期第一天的正規化值不一定等於 1.0，而是反映了相對於形成期首日的真實累積回報
3. Z-Score 計算不會因期別切換而產生人工跳躍

---

*分析來源：SSD Basic 配對交易滾動回測系統原始碼（忽略所有程式碼註解，僅以實際執行邏輯為準）*